# v0.5: The agent as a Hosted Agent on Microsoft Foundry

v0.4 finished the agent core. Everything it did ran inside a Python process that
someone started by hand, held a database connection that someone had already
authenticated, and answered a caller who was sitting in the same process.

v0.5 changes only where that core runs. The graph is untouched. What is new is a
protocol adapter, a container, and an identity of its own: the agent now runs as
a Microsoft Foundry Hosted Agent, reached over the Responses protocol, and
authenticates to the model and to Azure SQL as itself rather than as you.

That last sentence is the whole release. Most of what follows is about identity,
because moving from "runs as the developer" to "runs as itself" is where a hosted
agent actually breaks.

**Prerequisites:** `azd up` has provisioned the environment, `azd deploy` has
published the agent, `.env` is present, and `az login` has been run. Cells that
need the cloud report why they skipped instead of failing, so the notebook runs
end to end offline.

In [ ]:
from __future__ import annotations

import json
import time
import urllib.error
import urllib.request

from langchain_core.messages import HumanMessage

from enterprise_agents_on_foundry.agents.model import COGNITIVE_SERVICES_SCOPE, ModelInvocation
from enterprise_agents_on_foundry.agents.nodes import AgentDependencies, answer_question
from enterprise_agents_on_foundry.agents.state import (
    AgentInput,
    AgentOutcome,
    AgentOutput,
    GenerationDisposition,
    ModelCallMetadata,
    SqlGenerationResult,
)
from enterprise_agents_on_foundry.config.settings import load_settings, repository_root
from enterprise_agents_on_foundry.database.models import QueryRequest, QueryResult
from enterprise_agents_on_foundry.database.tool import QueryStatus, QueryToolResult
from enterprise_agents_on_foundry.database.validation import assert_read_only_sql
from enterprise_agents_on_foundry.errors import EaofError
from enterprise_agents_on_foundry.hosting.responses import (
    FAILED_MESSAGE,
    REJECTED_MESSAGE,
    UNSUPPORTED_INPUT_MESSAGE,
    UNSUPPORTED_QUESTION_MESSAGE,
    build_adapter_graph,
    extract_user_text,
    render_output,
)
from enterprise_agents_on_foundry.observability.measurements import MeasurementSet

measurements = MeasurementSet(release="v0.5")
settings = load_settings()

AGENT_NAME = "text-to-sql-agent"

print(f"project endpoint: {settings.azure_foundry_project_endpoint or '(unset)'}")
print(f"account endpoint: {settings.azure_foundry_account_endpoint or '(unset)'}")
print(f"model deployment: {settings.azure_model_deployment_name or '(unset)'}")
print(f"agent name:       {AGENT_NAME}")

## 1. Local graph invocation

The entry point has not changed since v0.3. `answer_question` takes dependencies
and an `AgentInput`, and returns an `AgentOutput`. No protocol appears anywhere in
that signature.

Scripted dependencies stand in for the model and the database so this cell is
deterministic and needs no cloud. They satisfy the same `AgentDependencies`
contract the production wiring satisfies.

In [ ]:
SCHEMA = "SalesLT.ProductCategory  Name nvarchar not null"
GOOD_SQL = "SELECT TOP (25) Name FROM SalesLT.ProductCategory"
UNSAFE_SQL = "DELETE FROM SalesLT.Product"


def result_rows(count: int) -> QueryResult:
    """A result carrying ``count`` rows, standing in for the database client."""
    return QueryResult(
        columns=("Name",),
        rows=tuple((f"Category {index}",) for index in range(count)),
        truncated=False,
        elapsed_ms=1.0,
        label="notebook",
    )


class ScriptedModel:
    """Replays generations so the graph runs without a model deployment."""

    def __init__(self, generations: list[SqlGenerationResult]) -> None:
        self._generations = list(generations)

    def draft(self, purpose: str, system: str, user: str) -> ModelInvocation[SqlGenerationResult]:
        generation = self._generations.pop(0) if len(self._generations) > 1 else self._generations[0]
        return ModelInvocation(metadata=ModelCallMetadata(purpose=purpose, latency_ms=0.0), value=generation)

    def write(self, purpose: str, system: str, user: str) -> ModelInvocation[str]:
        metadata = ModelCallMetadata(purpose=purpose, latency_ms=0.0)
        return ModelInvocation(metadata=metadata, value="There are four product categories.")


class ScriptedTool:
    """Replays tool results and records what was asked of the database."""

    def __init__(self, outcomes: list[QueryToolResult]) -> None:
        self._outcomes = list(outcomes)
        self.requests: list[QueryRequest] = []

    def execute(self, request: QueryRequest) -> QueryToolResult:
        self.requests.append(request)
        return self._outcomes.pop(0) if len(self._outcomes) > 1 else self._outcomes[0]


def scripted(
    generations: list[SqlGenerationResult],
    outcomes: list[QueryToolResult],
) -> tuple[AgentDependencies, ScriptedTool]:
    """Wire scripted stand-ins into the production dependency contract."""
    model = ScriptedModel(generations)
    tool = ScriptedTool(outcomes)
    deps = AgentDependencies(
        load_schema=lambda: SCHEMA,
        draft_sql=model.draft,
        write_answer=model.write,
        query_tool=tool,
    )
    return deps, tool


READY = SqlGenerationResult(disposition=GenerationDisposition.READY, sql=GOOD_SQL, rationale="lists the categories")
UNSAFE = SqlGenerationResult(disposition=GenerationDisposition.READY, sql=UNSAFE_SQL, rationale="does as asked")
FOUND = QueryToolResult(status=QueryStatus.SUCCESS, result=result_rows(4), elapsed_ms=2.0)
REFUSED = QueryToolResult(status=QueryStatus.REJECTED, error="not a read-only statement", elapsed_ms=0.4)

local_deps, _ = scripted([READY], [FOUND])
local_output = answer_question(local_deps, AgentInput(question="Which product categories exist?", max_rows=25))

print(f"type:    {type(local_output).__name__}")
print(f"outcome: {local_output.outcome.value}")
print(f"sql:     {local_output.sql}")
print(f"rows:    {local_output.row_count}")
print(f"answer:  {local_output.answer}")

## 2. Why the graph stays protocol-independent

The temptation when adopting a hosting protocol is to reshape the agent around
it, because the host wants a graph whose state carries a `messages` list and the
production graph does not have one.

This release refuses that trade. The dependency arrow points one way: `hosting/`
imports `agents/`, and nothing in `agents/` imports `hosting/`. The check below
is a static one over the source, so it fails loudly if that ever stops being true.

The reason is not tidiness. A core coupled to Responses would have to be changed
again for the next protocol, and every test of the core would need a protocol
fixture. Today the same `answer_question` serves the notebook, the tests, and the
host.

In [ ]:
agents_dir = repository_root() / "src" / "enterprise_agents_on_foundry" / "agents"
hosting_dir = repository_root() / "src" / "enterprise_agents_on_foundry" / "hosting"

core_to_protocol = [
    path.name for path in sorted(agents_dir.glob("*.py")) if "hosting" in path.read_text(encoding="utf-8")
]
protocol_to_core = [
    path.name for path in sorted(hosting_dir.glob("*.py")) if "agents" in path.read_text(encoding="utf-8")
]

print(f"agents/ modules referencing hosting: {core_to_protocol or 'none'}")
print(f"hosting/ modules referencing agents: {protocol_to_core}")
print()
print("dependency direction: hosting -> agents (one way)")

measurements.add(
    "core modules importing the protocol",
    len(core_to_protocol),
    category="design",
    note="agents/ must never import hosting/",
)

## 3. Responses request and response mapping

The adapter is the whole protocol boundary, and it does exactly two mappings.

**Inbound.** The host sends prior history followed by the current turn.
`extract_user_text` takes the latest human message and nothing else. Non-text
content, such as an image part, and a blank message both return `None` rather
than being coerced into a question.

**Outbound.** `render_output` maps each terminal `AgentOutcome` onto one
caller-facing sentence. Distinct outcomes stay distinct, but refusals and
failures render to fixed text, so an internal reason, a driver error, a token, or
a connection string cannot reach the caller through this boundary.

In [ ]:
print("inbound: latest human message wins")
history = [
    HumanMessage(content="How many customers are there?"),
    HumanMessage(content="Which product categories exist?"),
]
print(f"  {extract_user_text(history)!r}")
print(f"  image part -> {extract_user_text([HumanMessage(content=[{'type': 'image'}])])!r}")
print(f"  blank text -> {extract_user_text([HumanMessage(content='   ')])!r}")

print()
print(f"outbound: {'outcome':<24}rendered sentence")
for outcome, answer in (
    (AgentOutcome.SUCCEEDED, "There are four product categories."),
    (AgentOutcome.EMPTY, None),
    (AgentOutcome.CLARIFICATION_REQUIRED, None),
    (AgentOutcome.UNSUPPORTED, None),
    (AgentOutcome.REJECTED, None),
    (AgentOutcome.FAILED, None),
):
    rendered = render_output(AgentOutput(outcome=outcome, answer=answer))
    print(f"         {outcome.value:<24}{rendered}")

measurements.add(
    "terminal outcomes mapped to text",
    len(AgentOutcome),
    category="protocol",
    note="every outcome renders to one controlled sentence",
)

## 4. The local Hosted Agent server

`build_adapter_graph` wraps the production graph in a one-node `MessagesState`
graph, which is the shape `ResponsesHostServer` requires. Invoking that graph in
process is exactly what the host does per turn, minus HTTP.

`hosting/app.py` adds only the wiring: real settings, a real database client, a
real model, and `run(port=PORT)`. Running it locally is one command, and it is
the same entry point the container starts.

In [ ]:
adapter_deps, _ = scripted([READY], [FOUND])
adapter = build_adapter_graph(adapter_deps)

turn = adapter.invoke({"messages": [HumanMessage(content="Which product categories exist?")]})
reply = turn["messages"][-1]

print(f"adapter graph nodes: {len(adapter.get_graph().nodes)}")
print(f"reply type:          {type(reply).__name__}")
print(f"reply content:       {reply.content}")
print()
print("unsupported input never reaches the graph:")
empty_turn = adapter.invoke({"messages": [HumanMessage(content="   ")]})
print(f"  {empty_turn['messages'][-1].content}")
print()
print("start the same host locally with:")
print("  uv sync --extra hosting")
print("  uv run eaof-host          # serves POST /responses on PORT (default 8088)")

## 5. The container boundary

The image has to reproduce the layout the code expects, because
`config.settings.repository_root` resolves paths relative to the source tree. So
the package stays under `/app/src` and `database/queries` is copied beside it.

Three details in the Dockerfile are worth reading rather than skimming.

- `pyodbc` needs **ODBC Driver 18**, a system package rather than a wheel. That
  single `apt-get` layer is the largest contributor to image size.
- The image runs as an **unprivileged user**, `appuser`, uid 10001.
- Dependency resolution is **locked** (`uv sync --frozen`), so the image cannot
  silently drift from `uv.lock`.

The cache mounts that a local BuildKit build would use are deliberately absent:
ACR Tasks builds this image with the classic builder, which does not support
`RUN --mount`.

In [ ]:
dockerfile = (repository_root() / "Dockerfile").read_text(encoding="utf-8").splitlines()
interesting = ("FROM ", "WORKDIR", "COPY ", "USER ", "EXPOSE", "CMD", "ENV ")

for line in dockerfile:
    if line.startswith(interesting):
        print(line)

buildkit_mounts = [line for line in dockerfile if "--mount=type=cache" in line]
print()
print(f"BuildKit-only cache mounts: {len(buildkit_mounts)} (ACR Tasks uses the classic builder)")

measurements.add(
    "container image size",
    645,
    unit="MB",
    category="hosting",
    note="docker images, dominated by ODBC Driver 18 and the Python runtime",
)

## 6. `azure.yaml` configuration

One file describes the agent to azd: the protocol it speaks, the container
resources it gets, and the environment variables Foundry injects.

Two entries carry more weight than their size suggests.

`remoteBuild: true` builds the image inside ACR rather than locally. Only the
small source context crosses the network, which matters on any connection where
pushing a 645 MB image is unreliable.

The `env` block passes endpoints and names, and **no secrets**. There is no
connection string and no key, because every credential in this release is an
Entra token acquired at run time.

In [ ]:
azure_yaml = (repository_root() / "azure.yaml").read_text(encoding="utf-8")

print(azure_yaml.split("infra:")[0].strip())

secret_markers = [word for word in ("password", "secret", "key=", "connectionstring") if word in azure_yaml.lower()]
print()
print(f"secret-looking entries in azure.yaml: {secret_markers or 'none'}")

## 7. Deployment and versioning

`azd deploy` packages the image, publishes it to ACR, creates a new **agent
version**, and polls until that version reports `active`. Versions are immutable
and monotonic: the deployed agent is `text-to-sql-agent:2`, and the next deploy
produces version 3 rather than mutating 2.

Two behaviours are worth knowing before the first deploy.

The publish step needs `AZURE_AI_PROJECT_ID`, the ARM resource id of the Foundry
project. This release adds it as a Bicep output, because azd resolves the Hosted
Agent target from it and fails the publish step without it.

A hosted agent must publish its image to a container registry, so
`ENABLE_CONTAINER_REGISTRY` moves from `false` to `true` in this release.

In [ ]:
print("deploy:")
print("  azd provision                 # container registry and role assignments")
print("  azd deploy text-to-sql-agent  # build in ACR, publish, create a version")
print("  azd ai agent show text-to-sql-agent")
print()

main_bicep = (repository_root() / "infra" / "main.bicep").read_text(encoding="utf-8")
for name in ("AZURE_AI_PROJECT_ID", "AZURE_CONTAINER_REGISTRY_ENDPOINT"):
    present = f"output {name} " in main_bicep
    print(f"bicep output {name:<34} {'present' if present else 'MISSING'}")

measurements.add("deployed agent version", 2, category="hosting", note="immutable, monotonic per deploy")

## 8. Caller identity versus Hosted Agent instance identity

This is the section that explains most of the failures a first hosted deployment
produces, and it is the one genuinely new idea in the release.

Locally, `DefaultAzureCredential` resolves to **you**. Your `az login` account is
the Entra administrator on the SQL server and holds roles on the Foundry account,
so everything works and the agent's own permissions are never exercised.

In Foundry, the same `DefaultAzureCredential` resolves to the **agent instance
identity**: a distinct service principal named after the account, project, and
agent. It starts with no roles and no database user at all.

There is a third identity that is easy to confuse with it. The **user-assigned
managed identity** created by `infra/modules/identity.bicep` is the one the
original RBAC module grants roles to, and the hosted agent does **not** run as
it. Granting the user-assigned identity more permission has no effect on the
running agent.

The instance identity name carries no version suffix, so it survives a redeploy.
The blueprint identity carries the agent GUID and does not.

In [ ]:
identities = (
    ("caller (local)", "your az login account", "SQL Entra admin, Foundry roles", "n/a"),
    ("agent instance", "...-text-to-sql-agent-AgentIdentity", "granted in this release", "stable"),
    ("agent blueprint", "...-<guid>-AgentIdentityBlueprint", "none required", "per version"),
    ("user-assigned MI", "id-eaof-dev", "granted by Bicep, unused at run time", "stable"),
)

print(f"{'identity':<18}{'principal':<40}{'permissions':<38}across deploys")
print("-" * 118)
for label, principal, permission, lifetime in identities:
    print(f"{label:<18}{principal:<40}{permission:<38}{lifetime}")

print()
print("look it up with:")
print("  azd ai agent show text-to-sql-agent   # Instance Identity Principal ID")
print("  az ad sp show --id <principal-id> --query displayName -o tsv")

measurements.add(
    "distinct identities in the deployed system",
    len(identities),
    category="identity",
    note="only the agent instance identity authenticates at run time",
)

## 9. Entra-authenticated model access

The Foundry account is provisioned with `disableLocalAuth: true`, so there is no
API key to fall back on and no key to leak. `agents/model.py` builds a bearer
token provider over `DefaultAzureCredential` for the Cognitive Services scope.

That single design decision is why the agent identity needs an explicit role.
Inference on a keyless account requires **Cognitive Services OpenAI User** on the
Foundry account, and until it is granted the model call raises, the adapter
renders `FAILED_MESSAGE`, and the caller sees a generic error with no hint that
the cause was authorization.

In [ ]:
print(f"token scope:      {COGNITIVE_SERVICES_SCOPE}")
print(f"account endpoint: {settings.azure_foundry_account_endpoint or '(unset)'}")
print(f"api version:      {settings.azure_openai_api_version}")
print("local auth:       disabled on the Foundry account (no key exists)")
print()
print("required role for the agent instance identity:")
print("  Cognitive Services OpenAI User, scoped to the Foundry account")
print()
print("symptom when missing: every question returns")
print(f"  {FAILED_MESSAGE}")

## 10. Read-only Azure SQL access

The agent identity authenticates to Azure SQL with an Entra token, so it needs a
**contained database user**, created with `CREATE USER ... FROM EXTERNAL
PROVIDER`. There is no login and no password anywhere in this path.

It receives `db_datareader` and an explicit `DENY` on every write verb. That is
the durable control: the validator in `database/validation.py` is defence in
depth, and the database permission is what holds if the validator ever has a bug.

Reachability is worth separating from authentication here. The firewall rule
`AllowAllWindowsAzureIps` already admits Azure-hosted callers, so no firewall
change was needed for this release. A blocked network shows up as a timeout; what
a missing database user shows up as is `Login failed for user
'<token-identified principal>'`.

In [ ]:
grants = (
    ("CONNECT", "GRANT", "open a session with an Entra token"),
    ("db_datareader", "ROLE", "SELECT for schema discovery and queries"),
    ("INSERT / UPDATE / DELETE", "DENY", "explicit, survives role drift"),
    ("ALTER / EXECUTE", "DENY", "no DDL, no stored procedures"),
)

print(f"{'permission':<28}{'state':<8}why")
print("-" * 84)
for permission, state, why in grants:
    print(f"{permission:<28}{state:<8}{why}")

print()
print("applied by:")
print("  uv run python scripts/bootstrap_database.py --agent-identity-name <instance identity>")

print()
try:
    from enterprise_agents_on_foundry.database.connection import connect

    live_client = connect(settings)
    print(f"local connectivity: {live_client.health_check().summary}")
    live_client.close()
except EaofError as error:
    print(f"local connectivity skipped: {str(error).splitlines()[0]}")

measurements.add(
    "database roles held by the agent",
    1,
    category="security",
    note="db_datareader only, with writes explicitly denied",
)

## 11. Invoking the remote Responses endpoint

The deployed endpoint is derived from the project endpoint, so nothing here is
hard-coded. The caller authenticates with its own Entra token, which is a
different identity from the one the agent uses internally: you are authorized to
*call* the agent, and the agent is authorized to reach the model and the database.

This cell needs the deployed environment. Without it, it reports why it skipped.

In [ ]:
remote_ms = None
remote_answer = None


def ask_remote(question: str, timeout: int = 120) -> str:
    """Send one question to the deployed Responses endpoint and return the text."""
    from azure.identity import DefaultAzureCredential

    base = (settings.azure_foundry_project_endpoint or "").rstrip("/")
    url = f"{base}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1"
    token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token
    request = urllib.request.Request(  # noqa: S310
        url,
        data=json.dumps({"input": question}).encode("utf-8"),
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:  # noqa: S310
        payload = json.loads(response.read().decode("utf-8"))
    return payload["output"][0]["content"][0]["text"]


if not settings.azure_foundry_project_endpoint:
    print("skipped: AZURE_FOUNDRY_PROJECT_ENDPOINT is not set")
else:
    try:
        started = time.perf_counter()
        remote_answer = ask_remote("How many product categories are there?")
        remote_ms = round((time.perf_counter() - started) * 1000, 1)
        print(f"answer:  {remote_answer}")
        print(f"latency: {remote_ms} ms end to end")
    except (urllib.error.URLError, OSError, KeyError, IndexError) as error:
        print(f"skipped: {type(error).__name__}: {error}")

measurements.add(
    "remote responses latency",
    remote_ms,
    unit="ms",
    category="performance",
    note="end to end, includes two model calls and one query; varies widely",
)

## 12. Local versus hosted

The same graph, the same validator, and the same prompts run in both places. What
changes is who the process is and how it is reached.

Read the identity row first. It is the only row that caused a failure in this
release, and it is the row a local test can never exercise, because locally the
agent borrows your permissions.

In [ ]:
comparison = (
    ("entry point", "answer_question(deps, input)", "POST /responses"),
    ("identity", "your az login account", "agent instance identity"),
    ("model auth", "your Foundry roles", "Cognitive Services OpenAI User"),
    ("database auth", "you, the Entra admin", "contained user, db_datareader"),
    ("failure surface", "exception with a traceback", "one controlled sentence"),
    ("lifecycle", "process you started", "versioned, session-managed"),
    ("graph", "unchanged", "unchanged"),
)

print(f"{'concern':<18}{'local':<32}hosted")
print("-" * 82)
for concern, local, hosted in comparison:
    print(f"{concern:<18}{local:<32}{hosted}")

## 13. Expected failures and rejected writes

A write request is refused three times over, and the layers are independent.

The model classifies it as unsupported, so no SQL is drafted. If it drafted one
anyway, `assert_read_only_sql` rejects it before the driver sees it. If the
validator had a bug, `db_datareader` and the explicit `DENY` refuse it at the
database.

Every refusal reaches the caller as fixed text. That is deliberate: a caller who
learns *why* a statement was refused learns about the validator, and the next
attempt is better informed.

In [ ]:
print("layer 2, the validator, before any connection is opened:")
blocked = 0
for statement in (
    "DELETE FROM SalesLT.Customer;",
    "UPDATE SalesLT.Product SET ListPrice = 0;",
    "SELECT 1; DROP TABLE SalesLT.Customer;",
):
    try:
        assert_read_only_sql(statement)
        print(f"  ACCEPTED, which is a bug: {statement}")
    except EaofError as error:
        blocked += 1
        print(f"  rejected: {statement}  ({error})")

print()
print("through the protocol boundary, a rejected statement renders as:")
rejected_deps, rejected_tool = scripted([UNSAFE], [REFUSED])
rejected_turn = build_adapter_graph(rejected_deps).invoke({"messages": [HumanMessage(content="Delete every customer")]})
print(f"  {rejected_turn['messages'][-1].content}")

print()
print("the fixed sentences a caller can ever see:")
for sentence in (UNSUPPORTED_INPUT_MESSAGE, UNSUPPORTED_QUESTION_MESSAGE, REJECTED_MESSAGE, FAILED_MESSAGE):
    print(f"  {sentence}")

measurements.add(
    "unsafe statements blocked before execution",
    blocked,
    category="security",
    note="validator only; the database DENY is an independent second control",
)

## 14. Measurements and limitations

Recorded beside the release note. Test counts come from the release commit.

In [ ]:
measurements.add("offline tests", 290, category="quality", baseline=238)
measurements.add("offline test duration", 12.9, unit="s", category="quality", baseline=24.0)
measurements.add("integration tests", 19, category="quality", baseline=14)
measurements.add("hosting adapter tests", 17, category="quality", baseline=0)
measurements.add(
    "azure roles required by the agent identity",
    3,
    category="identity",
    baseline=0,
    note="OpenAI User, Monitoring Metrics Publisher, and the database user",
)

print(measurements.format_table())
written = measurements.write_json(repository_root() / "docs" / "releases" / "v0.5-measurements.json")
print()
print(f"written to {written.relative_to(repository_root())}")

## What this release still does not do

Each of these is deferred deliberately, not overlooked.

- **The identity grants are not reproducible.** The agent instance identity does
  not exist until the first deploy, so its three grants cannot live in the
  provisioning templates and were applied by hand. A fresh `azd up` in a new
  environment reproduces all three failures. A post-deploy hook is the fix.
- **No streaming.** The agent answers one turn at a time and returns the whole
  answer at once, so a long query looks like a pause.
- **No conversation memory.** The session and conversation ids are carried by the
  protocol, but each question is answered independently. Checkpointing is v0.8.
- **One replica, no scale evidence.** The container is provisioned with 0.5 vCPU
  and 1 GiB, and nothing here measures concurrency or cold start.
- **Latency is not yet attributed.** The end-to-end number mixes two model calls,
  one query, and session start-up, and this release does not separate them.
- **Failures are opaque by design, and that cut both ways.** Rendering every
  exception to one sentence protects the caller, but during this release it also
  masked an authorization failure as a generic error. Logging the exception type
  server-side, without the message, would keep the boundary and restore
  diagnosability.

Design notes: `docs/architecture/v0.5-hosted-agent.md`.
Release summary: `docs/releases/v0.5.md`.